# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to explore and analyze the FAIR^2 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) and made available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
We load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset via mlcroissant (will parse schema and prepare for access)
dataset = mlc.Dataset(croissant_url)

# Fetch metadata, print summary for context
meta_obj = dataset.metadata  # Do not treat as dict

print(f"Dataset Title: {meta_obj.name}\n")
print(f"Description: {meta_obj.description}\n")
print(f"Version: {getattr(meta_obj, 'version', 'N/A')}")


## 2. Data Overview
Inspect available record sets, their fields, and associated unique `@id`s as per the dataset schema.

All references use the Croissant `@id` fields to ensure portability and reproducibility.

In [ ]:
# List all record sets, their @id, and fields
recordset_overview = []

print("Available Record Sets:")
for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    rs_name = getattr(rs, 'name', '')
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    # List all fields
    print("  Fields:")
    for fld in rs.fields:
        # Each field has .id and .name
        print(f"    - Field @id: {fld.id} (name: {fld.name}, type: {fld.data_type})")
    print()
    recordset_overview.append({'id': rs_id, 'name': rs_name, 'fields': [fld.id for fld in rs.fields]})

## 3. Data Extraction
We demonstrate loading records from all available record sets into `pandas.DataFrame` objects for further analysis.

All field extraction is performed referencing only their `@id`s.

In [ ]:
# Load records for all available record sets into DataFrames
# Use the list of recognized RecordSet @id's

all_recordset_ids = [rs['id'] for rs in recordset_overview]
dataframes = dict()

for rs_id in all_recordset_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show DataFrame columns for the first record set
selected_rs_id = all_recordset_ids[0] if all_recordset_ids else None
if selected_rs_id:
    print(f"Data columns in record set {selected_rs_id}:")
    print(dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No record sets defined in the Croissant schema.")

## 4. Exploratory Data Analysis (EDA)
This section applies basic data processing steps: filtering, normalization, and grouping for demonstration.

All data selection and grouping uses field `@id`s.

In [ ]:
# For EDA, select a numeric field and a grouping field by their @id from first record set.

import numpy as np

if selected_rs_id is not None:
    df = dataframes[selected_rs_id]
    # Find a numeric field
    recordset = next(rs for rs in dataset.metadata.record_sets if rs.id == selected_rs_id)
    numeric_field_id = None
    group_field_id = None
    # Find numeric and group (categorical/string) fields
    for fld in recordset.fields:
        # common numeric types
        if fld.data_type in ["Float", "Integer", "Number"] and fld.id in df.columns:
            numeric_field_id = fld.id
            break
    for fld in recordset.fields:
        if fld.data_type in ["Text", "String"] and fld.id in df.columns and fld.id != numeric_field_id:
            group_field_id = fld.id
            break
    if numeric_field_id:
        # Prepare data
        field_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = field_vals.quantile(0.75) # upper quartile for filtering

        filtered_df = df[field_vals > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_numeric - filtered_numeric.mean()) / filtered_numeric.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field (if available) and show means
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Mean normalized {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df[[f"{numeric_field_id}_normalized"]].head())
    else:
        print("No numeric field found in record set for EDA.")
else:
    print("No record sets defined.")

## 5. Visualization
We visualize the distribution of the selected numeric field and its normalized version, and optionally a boxplot by group (if group field was found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id is not None and 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(12, 4))

    # Histogram of original and normalized
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Original distribution: {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
    plt.title(f"Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")

    plt.tight_layout()
    plt.show()

    # Optional: Boxplot of normalized value by group, if group field exists
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=f"{numeric_field_id}_normalized", data=filtered_df)
        plt.title(f"Normalized {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
We loaded, explored, and visualized the FAIR^2 dataset using the `mlcroissant` library:

- All access to entities was performed via their Croissant `@id`.
- We listed available record sets and their fields, loaded the records into dataframes, and performed normalization and grouping.
- Visualizations revealed the original and normalized distribution for a chosen numeric field.

To further analyze this data, repeat the workflow above with other record sets or customize EDA according to your research needs.